**Vanguard**  
**Jaydon Hylton (2210144)** , **Chadwick Cox (1800729)** , **Semoy Smith (1505625)**

# Part 3:  Structured Input/Output Validation with Pydantic

`AcademicAuditReport` uses typed fields and `Field(description=...)`. gpt-4.1-mini is bound with `.with_structured_output(..., method="json_schema")` so a free-form audit becomes validated JSON.

In [ ]:
%pip install -q azure-identity langchain-openai langchain-core python-dotenv pydantic

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI

load_dotenv(Path.cwd() / ".env")
try:
    from google.colab import userdata

    for name in (
        "AZURE_AI_FOUNDRY_ENDPOINT",
        "AZURE_AI_FOUNDRY_KEY",
        "AZURE_MODEL_NAME",
        "AZURE_OPENAI_API_VERSION",
    ):
        if not os.getenv(name):
            try:
                os.environ[name] = userdata.get(name)
            except Exception:
                pass
except ImportError:
    pass


def build_chat_model() -> AzureChatOpenAI:
    endpoint = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT", "").strip().rstrip("/")
    kwargs = {
        "azure_endpoint": endpoint,
        "azure_deployment": os.getenv("AZURE_MODEL_NAME", "gpt-4.1-mini").strip(),
        "api_version": os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21").strip(),
        "temperature": 0.0,
        "max_tokens": 256,
    }
    if os.getenv("AZURE_AUTH_MODE", "local").lower() == "identity":
        kwargs["azure_ad_token_provider"] = get_bearer_token_provider(
            DefaultAzureCredential(),
            "https://cognitiveservices.azure.com/.default",
        )
    else:
        kwargs["api_key"] = os.getenv("AZURE_AI_FOUNDRY_KEY")
    return AzureChatOpenAI(**kwargs)